# Bigode IA + Pipi IA — Colab sob demanda

Este notebook mantém **llama.cpp para texto** e **ComfyUI para imagem** separados. Nada é carregado até a célula correspondente ser executada. A T4 é usada por um motor de cada vez para evitar disputa de VRAM.

Venure · venure.com.br

In [ ]:
# 1. Configuração da sessão
import os, subprocess, sys, time, shutil, pathlib
ROOT = pathlib.Path('/content')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive')
TEXT_MODELS = DRIVE_ROOT / 'projetos/Cerebro/modelos'
IMAGE_MODELS = DRIVE_ROOT / 'projetos/Cerebro/IA Imagem'
LOCAL_MODELS = ROOT / 'modelos'
LOCAL_IMAGE_MODELS = ROOT / 'IA Imagem'
COMFY_ROOT = ROOT / 'ComfyUI'
TEXT_PORT, IMAGE_PORT = 8082, 8188
os.environ['PYTHONUNBUFFERED'] = '1'
print('Sessão pronta; os motores são opcionais.')

In [ ]:
# 2. Google Drive — arquivos persistem entre sessões
from google.colab import drive
drive.mount('/content/drive')
for p in (TEXT_MODELS, IMAGE_MODELS, LOCAL_MODELS, LOCAL_IMAGE_MODELS): p.mkdir(parents=True, exist_ok=True)
print('Texto:', TEXT_MODELS)
print('Imagem:', IMAGE_MODELS)

In [ ]:
# 3. Dependências mínimas
%pip -q install -U llama-cpp-python chromadb
if not COMFY_ROOT.exists():
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip -q install -r /content/ComfyUI/requirements.txt
CUSTOM = COMFY_ROOT / 'custom_nodes/ComfyUI-GGUF'
if not CUSTOM.exists():
    !git clone -q https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
print('Runtime instalado.')

In [ ]:
# 4. Copiar modelos para o disco local da sessão
TEXT_MODEL_FILE = os.environ.get('TEXT_MODEL_FILE', '')
def copiar_se_necessario(origem, destino):
    origem, destino = pathlib.Path(origem), pathlib.Path(destino)
    if not origem.exists(): return False
    if destino.exists() and destino.stat().st_size == origem.stat().st_size: return False
    tmp = destino.with_suffix(destino.suffix + '.part')
    shutil.copy2(origem, tmp); tmp.replace(destino); return True
if TEXT_MODEL_FILE:
    copiar_se_necessario(TEXT_MODELS / TEXT_MODEL_FILE, LOCAL_MODELS / TEXT_MODEL_FILE)
for arq in IMAGE_MODELS.glob('*.gguf'):
    copiar_se_necessario(arq, LOCAL_IMAGE_MODELS / arq.name)
print('GGUF locais:', [p.name for p in LOCAL_MODELS.glob('*.gguf')])
print('Auxiliares esperados: clip_l.safetensors e ae.safetensors')

In [ ]:
# 5. Ligar texto somente quando precisar
TEXT_MODEL = LOCAL_MODELS / TEXT_MODEL_FILE if TEXT_MODEL_FILE else None
if TEXT_MODEL and TEXT_MODEL.exists():
    subprocess.run("pkill -f 'llama_cpp.server --model' || true", shell=True)
    TEXT_PROC = subprocess.Popen([sys.executable, '-m', 'llama_cpp.server',
        '--model', str(TEXT_MODEL), '--host', '127.0.0.1', '--port', str(TEXT_PORT),
        '--n_ctx', '8192', '--n_gpu_layers', '-1', '--n_threads', '2'],
        stdout=open('/content/llama.log','w'), stderr=subprocess.STDOUT)
    print('Texto ligando:', TEXT_MODEL.name)
else:
    print('Texto não ligado. Defina TEXT_MODEL_FILE quando quiser usar o Bigode.')

In [ ]:
# 6. Ligar Pipi IA somente quando precisar
COMFY_PROC = subprocess.Popen([sys.executable, str(COMFY_ROOT / 'main.py'),
    '--listen', '127.0.0.1', '--port', str(IMAGE_PORT), '--lowvram', '--preview-method', 'auto'],
    cwd=str(COMFY_ROOT), stdout=open('/content/comfyui.log','w'), stderr=subprocess.STDOUT)
print('Pipi IA ligando na porta', IMAGE_PORT)
print('Aguarde a primeira carga do modelo antes de gerar.')

In [ ]:
# 7. Saúde e diagnóstico rápido
import urllib.request
def checar(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r: return r.status
    except Exception as e: return str(e)[:100]
print('Texto:', checar(f'http://127.0.0.1:{TEXT_PORT}/'))
print('Pipi:', checar(f'http://127.0.0.1:{IMAGE_PORT}/system_stats'))
print('Logs: /content/llama.log e /content/comfyui.log')

## Operação diária

Execute a célula 5 para o Bigode conversar ou a célula 6 para a Pipi criar imagens. Se a VRAM estiver apertada, encerre o outro processo antes de ligar o segundo motor. A interface local usa `/api/imagens` para mostrar componentes e `/api/imagens/gerar` para enviar o prompt.

O túnel deve ser atualizado a cada nova sessão. Nunca publique chaves no notebook.